# A session from LOBSTER files

[`documentation/from-lobster-files-to-a-session.md`](../documentation/from-lobster-files-to-a-session.md)
describes what LOBSTER writes, what the format permits that one day of data never shows, and
how a pair of files becomes a `MarketSession`. Every claim it makes is measured here, in the
section that makes it, with the code that measures it.

That is the skill this notebook is about, more than the format itself: meeting a large
unfamiliar dataset, stating what you expect of it, and reconciling the two. The pattern
repeats — an expectation from the specification, a measurement from the data, and a note
where they disagree. They disagree more often than one would like.

The data is not in the repository (`data/` is gitignored, 2.4 GB), so nothing below can be
re-run without it; the reference is written to be read without it.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pandera.errors as pe

from unito26.lob import frames, lobster
from unito26.lob.lobster_session import (
    LobsterMarketSession, coarsening_report, coarsening_totals, execution_gap_census,
    market_order_census, market_order_index, price_reversing_orders,
)
from unito26.lob.messages import GridDepth, SweepSize, TickGrid
from unito26.lob.statistics import SessionStatistics

pd.set_option("display.width", 130)

DATA = Path("..") / "data" / "lobster"
CENT = TickGrid(0.01)
UNIT = lobster.price_unit(CENT)
SPEC = SessionStatistics((GridDepth(1), GridDepth(5)), (SweepSize(100),), (60, 300))

PAIRS = [lobster.LobsterFiles.parse(p) for p in sorted(DATA.glob("*_message_*.csv"))]
AMZN = next(f for f in PAIRS if f.ticker == "AMZN" and f.reported_depth == 10)

## 0. Meeting the dataset

Before parsing anything: what the files are called, how large they are, and how many lines
they hold. The names alone fix the schema — ticker, day, requested window in milliseconds,
and the depth the book is reported to — so the number of columns is known before a byte is
read.

The two row counts are the thing to look at. They agree on every pair, and that agreement is
the only evidence for the alignment that can be gathered without parsing: the orderbook file
carries no clock, no sequence number and no key, so row $i$ is the state after message $i$
because of how the file was written and for no reason recoverable from its contents.

In [ ]:
census = lobster.file_census(PAIRS)
census.assign(Aligned=census["MessageRows"] == census["BookRows"])

Counting lines costs nothing next to parsing them, which is what makes it worth doing first.
The orderbook files are between five and thirty times the size of their message files, and
the largest is 1.45 GB.

In [ ]:
%time lobster._count_rows(DATA / "SPY_2012-06-21_34200000_37800000_orderbook_50.csv")

### What a window costs

The message file is the smaller of the pair and carries the only clock, so it is read whole
and used as the index into the other; only the rows the window covers are parsed. The bound
is set by the rows kept and not by where they sit, since skipping still streams the text.

In [ ]:
import time


def cost(files, window, label):
    started = time.perf_counter()
    messages, book = lobster.load_aligned(files, window)
    return {
        "case": label,
        "rows": len(book),
        "of": len(messages) if window is files.span else lobster._count_rows(files.orderbook_path),
        "seconds": round(time.perf_counter() - started, 2),
        "book MB": round(book.memory_usage(deep=True).sum() / 1e6, 1),
    }


spy = next(f for f in PAIRS if f.ticker == "SPY")
five_minutes = lobster.TradingWindow(35400.0, 35700.0)
pd.DataFrame([
    cost(AMZN, lobster.NASDAQ_REGULAR_HOURS, "AMZN d10, whole session"),
    cost(AMZN, five_minutes, "AMZN d10, five minutes"),
    cost(spy, five_minutes, "SPY d50, five minutes"),
]).set_index("case")

Five minutes of SPY at depth 50 is over a hundred megabytes of frame. Reading the whole file
peaks at several gigabytes, which is why every measurement below that needs only part of the
book asks for that part: `touch_prices` reads two of the 200 columns, and `padding_census`
sweeps the price columns in chunks.

---

## 1. The two files

For each ticker and day a **message** file and an **orderbook** file, one row of each per
event. The message file has six columns — time, type, order id, size, price, direction — and
prices are dollars times 10000. The orderbook file has $4 \times \text{LEVEL}$: ask price,
ask size, bid price, bid size, repeated outward from the touch.

Neither has a header, so the names come from the schema, which comes from the filename.

In [ ]:
print(AMZN.ticker, AMZN.day, "depth", AMZN.reported_depth)
print("span from the name (ms -> s):", AMZN.span)
print("orderbook twin:", AMZN.orderbook_path.name)
print("columns known before reading:", len(frames.lobster_book_columns(AMZN.reported_depth)))
print(frames.lobster_book_columns(AMZN.reported_depth)[:8], "...")

### The event types, and the ones no file contains

The ReadMe documents types 1, 2, 3, 4, 5 and 7. `LobsterEvent` also names 6, which is *not*
in the sample ReadMe and should be confirmed against LOBSTER's full documentation before
anything is asserted about it.

The census has a column for every documented type, including the ones the data does not
have. That is deliberate, and it is the argument of section 3 in one table: the columns come
from the format, so a type that never occurs is visibly absent rather than silently missing.

In [ ]:
lobster.event_census(PAIRS).set_index(["Ticker", "Depth"])

No type 6 and no type 7 anywhere in the sample. **Half the events are cancellations.** Most
posted orders are withdrawn rather than traded, which is what price-time priority produces: a
queue position has value, and the cheapest way to hold a good one is to post early and
withdraw when the market moves.

### `Direction` names the side that rested, not the side that traded

The ReadMe says it outright: the execution of a sell limit order is a buyer-initiated trade.
So the aggressor's direction is $-d$, and trade signing — a literature elsewhere — is exact
and free here.

The claim can be checked exactly rather than statistically. Price-time priority fills the
best price first, so a visible execution must print at the prior best price *on the side that
rested*. Every row, or the reading is wrong.

In [ ]:
raw = LobsterMarketSession.from_files(AMZN, SPEC, CENT, lobster.NASDAQ_REGULAR_HOURS)
off = raw.executions_off_the_touch()
visible = (raw.messages["Type"] == lobster.LobsterEvent.EXECUTION_VISIBLE).sum()
print(f"{visible} visible executions, {len(off)} of them not at the prior best on the resting side")

### Padding, and the trap in it

Where a side holds fewer than `LEVEL` occupied prices the remaining slots are padded: price
`+9999999999` on the ask, `-9999999999` on the bid, size 0 in both cases.

The two sentinels have **opposite signs**. A filter written for one lets the other through,
and one padded row moves a mean spread by $10^8$ ticks.

In [ ]:
aapl50 = next(f for f in PAIRS if f.ticker == "AAPL" and f.reported_depth == 50)
opening = lobster.TradingWindow(34200.0, 34500.0)
touch = lobster.touch_prices(aapl50, opening)
deep = lobster.load_orderbook(aapl50.orderbook_path, aapl50.reported_depth).iloc[: len(touch)]

quoted = (deep["AskPrice50"] != frames.ASK_PADDING) & (deep["BidPrice50"] != frames.BID_PADDING)
level50 = (deep["AskPrice50"] - deep["BidPrice50"]) / UNIT
plausible = deep["BidPrice50"] > 0  # cleans the bid correctly and the ask not at all
pd.DataFrame({
    "mean level-50 spread in ticks": [
        level50.mean(), level50[plausible].mean(), level50[quoted].mean(),
    ]
}, index=["as written", "after BidPrice50 > 0", "after both sentinels"])

### Trading halts, which no sample file contains

A halt writes a type-7 message with `Size` and `OrderID` zero, `Direction` `-1`, and the
`Price` column carrying a **status code**: `-1` halted, `0` quoting resumed, `+1` trading
resumed. On these rows `Price` is not a price, `Direction` names no side and `Size` is not a
quantity, and the orderbook rows facing them duplicate the preceding state.

Nothing in the shipped sample exercises this, so the file has to be written. `write_pair` is
the inverse of `load_aligned` and validates nothing on the way out, which is exactly what
makes it useful here: the cases worth showing are the ones a loader would refuse.

In [ ]:
import tempfile

WORKSHOP = Path(tempfile.mkdtemp())


def constructed(name, messages, book):
    files = lobster.LobsterFiles.parse(WORKSHOP / name)
    columns = frames.lobster_message_file_columns()
    depth = files.reported_depth
    return lobster.write_pair(
        files,
        pd.DataFrame(messages, columns=columns),
        pd.DataFrame(book, columns=frames.lobster_book_columns(depth)),
    )


STATE = [100200, 120, 100000, 100, frames.ASK_PADDING, 0, 99900, 10]
halted = constructed(
    "HALT_2012-06-21_34200000_57600000_message_2.csv",
    [
        (36023.0, 7, 0, 0, -1, -1),
        (36323.0, 7, 0, 0, 0, -1),
        (36723.0, 7, 0, 0, 1, -1),
    ],
    [STATE, STATE, STATE],
)
lobster.load_messages(halted.messages_path)

The loader admits it, because the schema was written against the format: `OrderID`
unconstrained, `Price` down to `-1`, `Size` down to `0`. Every one of those bounds looks
wrong until a halt arrives.

---

## 2. Levels are occupied levels

`LEVEL` counts **occupied** prices, not positions on the tick grid: level $k$ is the $k$-th
price carrying size, however far from the touch it sits. The notes' $I^n$ counts grid
positions instead, and the two coincide only on a book with no holes.
[`grid-levels-and-lobster-levels.md`](../documentation/grid-levels-and-lobster-levels.md) is
the reference for the difference.

The point worth making here is that the difference is **invisible in the output**. Both
quantities stay in $[-1, 1]$, both move with the market, and nothing about a column of
numbers says which one it is.

In [ ]:
session = raw.coarsened(True)
holes = session.stats[["BidGapCount", "AskGapCount"]]
print(f"rows whose reported levels are not contiguous on the grid: "
      f"{(holes.sum(axis=1) > 0).mean():.1%}")

grid = session.stats["QueueImbalance5"]
columns = session.column_sliced_imbalance(GridDepth(5))
pd.DataFrame({"I^5 over the grid": grid.describe(), "the first five columns": columns.describe()})

---

## 3. What the format permits, and what one day of data happens to show

The constraints below are the ones a reader writes first. Each is wrong, and they fail in two
different ways — which is the distinction worth keeping.

In [ ]:
lobster.constraint_census(raw.messages, UNIT).set_index("Constraint")

`OrderID > 0` fails on day one: hidden executions carry no order id, and neither does a halt.
That teaches only that the specification is worth reading.

The other three are the interesting ones. `Price >= 0` and `Size > 0` pass every shipped file
and reject a halt — which is why the halt had to be constructed above. `Price % 100 == 0`
passes every visible execution and rejects some hidden ones, because hidden prints are
sub-penny: the tick grid is a property of the **lit** book, not of the feed.

In [ ]:
sub_penny = raw.messages[raw.messages["Price"] % UNIT != 0]
sub_penny[["Time", "Type", "Size", "Price", "Direction"]].head(4)

And the fourth, which is not about a message at all: **the spread is positive on every row of
every file**. That is a property of one day. Nothing in the format promises it, and a schema
asserting it would pass every test written against this sample.

Answered over all 3.5 million book states by reading two of each file's columns.

In [ ]:
%time crossing = lobster.touch_census(PAIRS, lobster.NASDAQ_REGULAR_HOURS, CENT)
crossing.set_index(["Ticker", "Depth"])

### Padding is real, and it is asymmetric

At depth 10 none of the samples pads at all; at depth 50 all three do. Padding is an opening
artefact — the book has not yet filled to fifty levels — so a window that does not start at
09:30 contains none, and an assertion about padding written over such a window is vacuous
rather than passing.

Note which side truncates on SPY. The ask is the side one tests first.

In [ ]:
pd.concat(
    [lobster.padding_census(f, f.span).assign(Ticker=f.ticker)
     for f in PAIRS if f.reported_depth == 50]
).groupby(["Ticker", "Side"]).agg(
    ShallowestLevel=("Level", "min"), DeepestLevel=("Level", "max"),
    RowsAtTheDeepest=("Rows", "last"), FirstTime=("FirstTime", "min"),
    LastTime=("LastTime", "max"),
)

---

## 4. The clock

Timestamps are decimal seconds after midnight. The ReadMe promises at least milliseconds and
up to nanoseconds. The file carries more than that.

In [ ]:
lobster.clock_census(AMZN.messages_path).set_index("Decimals")

Twelve decimals on two rows, finer than the documentation admits and finer than a nanosecond
key keeps. Also a long tail of coarser rows, which the reference's own summary omits.

### Ties are ordinary

Times are non-decreasing and never decrease. They are also **not unique**.

In [ ]:
stamps = raw.messages["TimeNanoseconds"].to_numpy()
repeats = pd.Series(stamps).value_counts()
print(f"{(np.diff(stamps) == 0).sum()} tied steps, "
      f"{(np.diff(stamps) == 0).mean():.1%} of the file")
print(f"largest number of messages at one instant: {repeats.max()}")

Two consequences. A rolling window must decide what a tie means, and the choice here is
causal: the window $(t-w,\,t]$ is closed on the right at the current row, so two rows sharing
a timestamp get different windows and row $i$ does not see row $i+1$.

And an index schema must **not** declare the timestamp unique. That is the constraint a
reader adds next, and section 5 says where the ties come from.

In [ ]:
import pandera.pandas as pa

careless = pa.DataFrameSchema(
    {"AskPrice1": pa.Column("Int64", coerce=True)},
    index=pa.Index(float, name="TimeStamp", unique=True),
)
try:
    careless.validate(session.lobster_book[["AskPrice1"]])
except pe.SchemaError as refused:
    print(str(refused).splitlines()[0])

### Nothing may group on the float

Equality on a float is exact-bit equality, and here it works: every instant the text
distinguishes survives the parse. But it works *because the clock counts from midnight*,
which keeps the exponent small.

The two columns to read against each other are the spacing float64 can represent at that
magnitude and the closest pair of instants the file actually holds. Shift the same clock to
the Unix epoch and the first crosses the second.

In [ ]:
origins = {"midnight": 0, "Unix epoch": int(pd.Timestamp(AMZN.day).value)}
pd.concat([
    lobster.clock_origin_census(lobster.load_messages(f.messages_path), origins)
    .assign(Ticker=f.ticker)
    for f in (AMZN, next(f for f in PAIRS if f.ticker == "INTC"), spy)
]).set_index(["Ticker", "Origin"])

From midnight the representable spacing is four orders of magnitude below the closest pair of
instants, and nothing can merge. From the Unix epoch it is **larger** than that closest pair,
and instants do merge — none on AMZN, but tens on the busier files.

So the safety is a property of the encoding and not of the format, and how much of it there
is depends on how busy the ticker is. The *key* is therefore an exact integer built from the
text by splitting on the decimal point, never by scaling the parsed float, and
`TimeNanoseconds` is the only column anything in the package groups on.

---

## 5. One order, several rows

LOBSTER records the **executions of resting orders**, not trades. An incoming order that
consumes $k$ of them produces $k$ message rows and $k$ orderbook rows, all sharing one
timestamp, with the intermediate book states visible between them.

Take the worked example's book — best bid 100 at 1000 — and suppose, as the aggregate book
cannot, that the 100 is five resting orders of 20. A sell of 55 arrives.

In [ ]:
split = constructed(
    "SPLT_2012-06-21_34200000_57600000_message_2.csv",
    [
        (34201.5, 4, 11, 20, 100000, 1),
        (34201.5, 4, 12, 20, 100000, 1),
        (34201.5, 4, 13, 15, 100000, 1),
    ],
    [
        [100200, 120, 100000, 80, frames.ASK_PADDING, 0, 99900, 10],
        [100200, 120, 100000, 60, frames.ASK_PADDING, 0, 99900, 10],
        [100200, 120, 100000, 45, frames.ASK_PADDING, 0, 99900, 10],
    ],
)
demo = LobsterMarketSession.from_files(split, SPEC, CENT, lobster.NASDAQ_REGULAR_HOURS)
pd.concat([
    demo.messages[["Time", "Type", "OrderID", "Size", "Price", "Direction"]],
    demo.book[["BidPrice1", "BidSize1"]],
], axis=1)

Three rows where a fold writes one, and the bid size passing through 80 and 60 — book
configurations that exist *inside* the matching of a single incoming order.

### How often, and grouped how

Which fills belong to one order is a choice, and the three reasonable answers do not agree.
Contiguity in the file is the right one — the fills of one aggressive order are consecutive
in the sequence, so a `groupby` on the instant joins two orders that merely arrived together —
and interior hidden prints are absorbed, because one aggressor can take lit and hidden
liquidity in a single sweep.

In [ ]:
tickers = [f for f in PAIRS if f.reported_depth == 10 and f.ticker != "MSFT"]
counted = market_order_census(tickers)
counted.assign(
    MultiFillShare=(counted["MultiFill"] / counted["MarketOrders"]).map("{:.1%}".format),
    FillsInsideShare=(counted["FillsInside"] / counted["Executions"]).map("{:.1%}".format),
    QueueSplitShare=(counted["QueueSplits"] / counted["MultiFill"]).map("{:.0%}".format),
).set_index(["Ticker", "Convention"])

A fifth to a half of market orders take more than one fill, and on INTC 86% of all executions
sit inside one.

The last column is the one that matters. A multi-fill order is either a **level walk** — the
aggressor consumes several prices, which an aggregate book could report one level at a time
if it chose — or a **queue split**, several resting orders at one price, which no sequence of
aggregate states determines, because the aggregate book does not know a level of 100 is five
orders of 20. Four fifths are queue splits, so the granularity gap is overwhelmingly the part
an aggregate book cannot reach even in principle.

### Why grouping on equality is safe here

Within a split the instant is *exactly* equal, and the nearest pair of same-price same-side
executions that is not shares no instant by a margin of microseconds. There is no ambiguous
middle.

In [ ]:
execution_gap_census(raw.messages).set_index("Adjacency")

### The hole in the grouping, and how weak the evidence about it is

Two genuinely distinct aggressive orders arriving in the same nanosecond on the same side
merge into one, and the file carries no aggressor id to separate them. One signature is
visible: an aggressive order walks away from the mid and never back, so a price path that
reverses is two orders read as one.

In [ ]:
pd.DataFrame([
    {"Ticker": f.ticker, "reversing orders": len(price_reversing_orders(
        lobster.load_messages(f.messages_path)))}
    for f in tickers
]).set_index("Ticker")

None anywhere — and that establishes very little. Two orders each sweeping monotonically are
invisible to this test, so an empty answer is consistent with the hole being empty and does
not demonstrate it. Worth stating as a limitation rather than as a result.

---

## 6. The pipeline, and the schema at each stage

Every frame that crosses a function boundary is declared before any data is read and
validated on the way out. There are several shapes and no two are interchangeable, even
where their columns agree.

In [ ]:
pd.DataFrame([
    ("LobsterFiles.parse", "the pair, the depth, the span", "—", "reads nothing"),
    ("load_messages", "the message file, plus the exact clock", "lobster_message_schema", "positional"),
    ("load_orderbook", "the book states as written", "lobster_orderbook_file_schema", "positional"),
    ("load_aligned", "both, cut to a TradingWindow", "the two above", "positional"),
    ("LobsterMarketSession", "the pair as the files hold it", "the two above", "positional"),
    ("  .stats_from_frame", "statistics per file row", "positional_statistics_schema", "positional"),
    ("  .coarsened", "lobster_book", "session_book_schema", "TimeStamp"),
    ("MarketSession.stats_from_frame", "stats", "statistics_schema", "TimeStamp"),
], columns=["stage", "what it produces", "schema", "index"]).set_index("stage")

**The schema is fixed before the file is opened**, `LEVEL` being in the name. `read_csv` is
then *told* the names and the dtypes. Inference agrees with the declaration on every shipped
file, so this fixes nothing today; what it does is turn a future malformed row into an
exception instead of a silently widened column.

**But declaring a dtype is not the same as narrowing one.** `Type` and `Direction` are read
as `int64`, not the `int8` that would obviously hold them, because a narrow integer
*launders* corruption rather than catching it.

In [ ]:
corrupt = constructed(
    "CRPT_2012-06-21_34200000_57600000_message_2.csv",
    [(34200.5, 260, 7, 21, 100000, 1)],
    [STATE],
)
narrow = pd.read_csv(
    corrupt.messages_path, header=None,
    names=frames.lobster_message_file_columns(), dtype={"Type": "int8"},
)
print("a Type of 260 read as int8 becomes:", narrow["Type"].iloc[0])
try:
    lobster.load_messages(corrupt.messages_path)
except pe.SchemaError as refused:
    print("read as int64:", str(refused).splitlines()[0])

A `Type` of 260 wraps to 4 — a valid visible execution — inside `read_csv`, and any membership
check downstream then passes it. The declaration meant to catch the corruption is what hides
it.

**Positional and clocked frames are different schemas**, and the nullable extension dtype is
why the file schema is the plain one: coercing replaces one contiguous integer block with one
masked column per field.

In [ ]:
plain = lobster.load_orderbook(AMZN.orderbook_path, AMZN.reported_depth)
pd.DataFrame({
    "megabytes": [
        plain.memory_usage(deep=True).sum() / 1e6,
        plain.astype("Int64").memory_usage(deep=True).sum() / 1e6,
    ],
    "blocks": [len(plain._mgr.blocks), len(plain.astype("Int64")._mgr.blocks)],
}, index=["int64, as the file schema declares", "Int64, as a session frame holds it"])

**Alignment is assumed, not verified.** The row counts are compared, which catches a file
truncated or extended at either end. Nothing can catch a row missing from the *middle*: the
orderbook file has no key, so an off-by-one yields a complete session with every statistic
finite and every timestamp on the wrong book state.

In [ ]:
short = constructed(
    "GONE_2012-06-21_34200000_57600000_message_2.csv",
    [(34200.0 + i, 1, i + 1, 10, 100000, 1) for i in range(4)],
    [[100200, 120, 100000, 100 - 10 * i, frames.ASK_PADDING, 0, 99900, 10] for i in range(4)],
)
book = pd.read_csv(short.orderbook_path, header=None)
book.drop(index=1).to_csv(short.orderbook_path, header=False, index=False)  # a row from the middle
try:
    lobster.load_aligned(short, lobster.NASDAQ_REGULAR_HOURS)
except ValueError as refused:
    print("caught, because the counts now disagree:", refused)

book.drop(index=1).to_csv(short.orderbook_path, header=False, index=False)
pd.read_csv(short.orderbook_path, header=None, nrows=3)  # and were a row appended, nothing would

### Units: the conversion that rescales everything and fails nothing

Prices are integer **tick counts** everywhere inside the package; LOBSTER counts
ten-thousandths of a dollar. The bridge is one integer — how many of the file's units make
one tick — and the caller never states it. It states the **tick size**, which is a fact about
the instrument.

In [ ]:
print("a cent tick is", lobster.price_unit(TickGrid(0.01)), "of LOBSTER's units")
print("LOBSTER's own unit is", lobster.price_unit(TickGrid(0.0001)))
try:
    lobster.price_unit(TickGrid(0.000001))
except ValueError as refused:
    print(refused)

The failure mode is why this matters: a wrong unit does not raise. It rescales every price,
spread, mid-price and sweep cost the session reports — by a factor that **cancels in any
round trip through the same constant**, so a test that writes a book out and reads it back
cannot detect one, however carefully it is written.

In [ ]:
wrong = LobsterMarketSession.from_files(
    AMZN, SPEC, TickGrid(0.0001), lobster.TradingWindow(34200.0, 34260.0)
).coarsened(True)
right = LobsterMarketSession.from_files(
    AMZN, SPEC, CENT, lobster.TradingWindow(34200.0, 34260.0)
).coarsened(True)
pd.DataFrame({
    "tick size 0.01": right.stats[["Spread", "MidPrice"]].mean(),
    "tick size 0.0001": wrong.stats[["Spread", "MidPrice"]].mean(),
})

Both are internally consistent, both round-trip perfectly, and one of them is a hundred times
the other. The grid itself is checked once, at load, and refused with the offending price.

In [ ]:
crooked = constructed(
    "SKEW_2012-06-21_34200000_57600000_message_2.csv",
    [(34200.5, 1, 7, 21, 100050, 1)],
    [[100200, 120, 100050, 100, frames.ASK_PADDING, 0, 99900, 10]],
)
try:
    LobsterMarketSession.from_files(crooked, SPEC, CENT, lobster.NASDAQ_REGULAR_HOURS)
except ValueError as refused:
    print(refused)

---

## 7. Coarsening: from a pair of files to a session

A `MarketSession` has one row per aggressive order and one per every other message. A LOBSTER
pair does not. The two frames carry the same columns and count different things, and section
5 measured the difference.

The resolution is a transformation, and the two shapes are two types so that it cannot be
skipped by accident. A market order keeps the **last** row of its block — the state after the
whole order finished matching, which is the state a fold writes — and its fills are summed
onto it.

In [ ]:
coarse = raw.coarsened(True)
print(f"{len(raw.book)} rows of file -> {len(coarse.lobster_book)} rows of session")
try:
    frames.session_book_schema(raw.reported_depth).validate(raw.book)
except pe.SchemaError as refused:
    print("the pair is refused by the session schema:", str(refused).splitlines()[0])
try:
    frames.lobster_orderbook_file_schema(raw.reported_depth).validate(coarse.lobster_book)
except pe.SchemaError as refused:
    print("and the session by the file schema: ", str(refused).splitlines()[0])

That mutual refusal is the point. The reference observes that a schema pins the fields and
cannot pin what a *row* is; giving the two shapes two indices is how much of that gap a schema
can be made to close.

### What survives, and what it costs

In [ ]:
coarsening_totals(raw, coarse).set_index("Quantity")

Shares, traded value and the session VWAP are equal to the bit: a market order's fills are
additive and every fill belongs to exactly one order. What changes is the row count, and the
mean — and that one is the trap LOBSTER's own demo names. The mean of the fine column is the
mean **execution**; the mean of the coarse one is the mean **trade**; nothing in the file
marks the difference.

Statistic by statistic, with what the theory promised beside what happened:

In [ ]:
report = coarsening_report(raw, coarse)
report[report["RowsDiffering"] > 0].set_index("Statistic")

`Comparison` says how the fine side was combined before the two were compared, and it is not
a detail: the order flow contribution of a market order is the sum of its fills'
contributions, and comparing the *last* fill instead would report a difference on every
multi-fill order while measuring nothing.

Read against the counts of section 5, the increment row is exact:

In [ ]:
orders = raw.market_orders()
price = raw.messages["Price"].to_numpy()
lit = (raw.messages["Type"] == lobster.LobsterEvent.EXECUTION_VISIBLE).to_numpy()
fills = np.bincount(orders[orders >= 0])
walks = [o for o in np.flatnonzero(fills > 1)
         if len(np.unique(price[(orders == o) & lit])) > 1]
print("multi-fill orders:", int((fills > 1).sum()), " of them level walks:", len(walks))
print("rows where OrderFlowContribution differs:",
      int(report.set_index("Statistic").loc["OrderFlowContribution", "RowsDiffering"]))

The order flow contribution survives **every** queue split and **no** level walk, and that is
a theorem rather than an observation. An execution touches only the resting side, so writing
that side's touch as $(P_j, S_j)$ and summing the contributions over the fills,

$$\sum_n e_n \; - \; e(\text{first},\ \text{last}) \;=\; S_{\text{end}} - \sum_i S_{j_i},$$

the sum running over the fills at which the touch price moved. A queue split moves it at most
once, and when it does the level emptied on the last fill, so the two terms cancel. A walk
fills again after emptying a level, which strictly reduces the size at the new touch, so the
difference cannot vanish.

Nothing the theory guarantees ever differs:

In [ ]:
print("guaranteed and differing:",
      int((report["Guaranteed"] & (report["RowsDiffering"] > 0)).sum()))
print("not guaranteed and identical anyway:",
      report.loc[~report["Guaranteed"] & (report["RowsDiffering"] == 0), "Statistic"].tolist())

And the converse fails, which is the more interesting half. Every VWAP comes through
unchanged without being promised anything: it is a ratio of two sums each additive over the
fills, and no argument from what the statistic *depends on* reaches that. A report whose two
columns agreed everywhere would be one that had learnt nothing.

**What is discarded on purpose.** The file knows a level of 100 was five orders of 20; the
coarse session does not. That is exactly what an aggregate book cannot represent, and giving
it up is the price of a session whose rows mean what the simulator's rows mean. The other
direction — refining our own book to emit one event per resting order consumed — is the
identity-carrying rung of the ladder, and it is a different piece of work.